# Get to Know a Dataset: LAFC-Evict

**Dataset:** LAFC-Evict: Learning-Augmented Cache Eviction Dataset (v0.3)  
**Registry of Open Data on AWS entry:** `<PENDING_REGISTRY_LANDING_PAGE_URL>` (not live yet — this repository is a private AWS Open Data review package; no public Registry pull request has been opened)  
**Public AWS S3 dataset used for this notebook:** `s3://lafc-evict-open-data/` (anonymous HTTPS: https://lafc-evict-open-data.s3.us-west-2.amazonaws.com/)  
**Also available on Hugging Face:** https://huggingface.co/datasets/SoroushVahidi/lafc-evict

## Dataset introduction

LAFC-Evict is a derived, tabular research dataset of counterfactual supervision
labels for learning-augmented cache-eviction research. Each row describes one
*candidate object* considered for eviction at one full-cache-miss decision
point, together with engineered state/candidate features and a label describing
what would happen to the cache miss rate over a short future horizon if that
specific candidate were evicted. The released rows are entirely **derived**:
they are computed by the dataset author from third-party, CC0-1.0 Wikimedia
pageview data, and no raw Wikimedia rows or raw page titles are redistributed.

## Learning objectives

By the end of this notebook you will be able to:

1. Load a small, representative slice of LAFC-Evict without downloading the full file.
2. Inspect the schema and identify the columns that matter for a candidate-eviction-prediction task.
3. Reproduce one simple exploratory analysis (loss by cache capacity) and one visualization.
4. Understand what the main `y_loss` / `y_value` label does and does not mean.
5. See one worked research question this dataset already supports, and one open question it does not yet answer.

## How have you organized your dataset?

The v0.3 release ships as **two independent Parquet files**, one per
configuration, each holding a single flat `train` split (no directory
hierarchy inside a configuration):

| Configuration | Rows | Contents |
|---|---|---|
| `cross_family_evict_value_v1` | 11,178,496 | Finite-horizon eviction-loss/value labels (`y_loss`, `y_value`) |
| `objective_ablation_scalar` | 11,178,496 | Eviction-loss plus *censored* next-arrival and reuse-distance labels |

This notebook uses `cross_family_evict_value_v1` throughout.

The dataset is now published on the AWS Open Data public S3 bucket,
following the layout recommended in the AWS Open Data onboarding handbook
(top-level `data/`, `metadata/`, and `docs/` prefixes): `s3://lafc-evict-open-data/`
(anonymous HTTPS: `https://lafc-evict-open-data.s3.us-west-2.amazonaws.com/`).
This notebook reads directly from that public bucket over anonymous HTTPS
below — no AWS credentials are required. The dataset's original Hugging
Face home (`https://huggingface.co/datasets/SoroushVahidi/lafc-evict`)
remains available as an alternate public access path with the same
underlying Parquet bytes.


## What data formats are present in your dataset?

**Apache Parquet only.** No CSV, JSON, or other flat-file mirror is
published. Parquet was chosen because it is columnar (cheap to read a
subset of columns), splittable by row group (cheap to read a subset of
rows without downloading the whole file — exactly what this notebook
does below), and natively supported by `pandas`, `polars`, `pyarrow`,
Amazon Athena, and AWS Glue.


## Can you show us an example of downloading and loading data?

Because this release is 11.2M rows (~50-100MB per config as Parquet), we
avoid downloading the whole file for a 101-level tutorial. Instead we read
the remote Parquet file's footer metadata (a few hundred KB) over HTTP
range requests, then pull only a handful of **row groups** spread across
the file, so the sample still spans the dataset's real value diversity
(this dataset happens to be laid out with long contiguous runs of a single
`capacity` value, so reading only the first N rows sequentially would
under-represent capacities 64 and 128 -- reading spread-out row groups avoids
that trap).

**Requirements:** `pyarrow`, `fsspec` (for HTTP range reads), `pandas`. All
three are common data-science-stack packages; none are AWS-specific.
`requirements.txt` line: `pyarrow>=14 fsspec>=2024.2 pandas>=2.0 matplotlib>=3.7`

The cell below reads directly from the real, public AWS S3 bucket created
in Step 5 (`s3://lafc-evict-open-data/`) over anonymous HTTPS — no AWS
account or credentials are required. The same Parquet bytes are also
available from Hugging Face; that alternate URL is included, commented
out, below. The row-group-sampling logic is unchanged either way.


In [ ]:
import fsspec
import pyarrow.parquet as pq
import pandas as pd

# Public AWS Open Data S3 bucket (anonymous HTTPS, no AWS credentials required).
PARQUET_URL = (
    "https://lafc-evict-open-data.s3.us-west-2.amazonaws.com/"
    "data/cross_family_evict_value_v1.parquet"
)

# Alternate public access path (same underlying bytes), for reference:
# PARQUET_URL = (
#     "https://huggingface.co/datasets/SoroushVahidi/lafc-evict/"
#     "resolve/main/data/cross_family_evict_value_v1.parquet"
# )

fs = fsspec.filesystem("https")
remote_file = fs.open(PARQUET_URL, "rb")
parquet_file = pq.ParquetFile(remote_file)

print("row groups:", parquet_file.num_row_groups)
print("total rows :", parquet_file.metadata.num_rows)
print("columns    :", parquet_file.schema_arrow.names)


## Schema inspection

The columns fall into four families. Full definitions live in the
dataset's `metadata/schema.json` and in the accompanying open-data
documentation (`../documentation/README.md`, section 7):

- **Decision identifiers:** `decision_id`, `decision_t`, `capacity`, `horizon`, `split`, `trace_family`
- **Candidate identity:** `candidate_page_id` (internal), `object_id_public` (public pseudonym — use this one)
- **Engineered features:** `candidate_predictor_score`, `candidate_lru_score`, `*_gap_to_*`, `cache_bucket_*`, `recent_candidate_*`
- **Labels (this configuration):** `y_loss` (finite-horizon counterfactual LRU-continuation miss count), `y_value = -y_loss`

**Important:** `trace_family` is constant `"wiki2018"` in this release —
despite the configuration's name (`cross_family_evict_value_v1`), it does
**not** contain multiple trace families. This is a known naming artifact
inherited from an internal multi-family pipeline; always trust
`trace_family`, never `source_fold` (also constant, but for an unrelated
internal reason), to determine data provenance.

In [ ]:
# Pull a handful of row groups spread across the file so the sample still
# spans the real distribution of `capacity` values (see note above).
cols = [
    "trace_family", "capacity", "horizon", "decision_id", "object_id_public",
    "split", "y_loss", "y_value", "predictor_lru_disagree",
    "candidate_predictor_score", "candidate_lru_score",
]

n_groups = parquet_file.num_row_groups
sample_row_group_idxs = [
    int(n_groups * frac) for frac in (0.0, 0.2, 0.5)
]  # spans capacity=32, 64, 128 in this file's layout

frames = []
for idx in sample_row_group_idxs:
    table = parquet_file.read_row_group(min(idx, n_groups - 1), columns=cols)
    frames.append(table.to_pandas().head(800))

sample_df = pd.concat(frames, ignore_index=True)
print("sample rows:", len(sample_df))
sample_df["capacity"].value_counts()


## A picture is worth a thousand words

Two views of the same 2,400-row stratified sample: the distribution of the
`candidate_lru_score` feature (which has real, non-degenerate variance),
and mean `y_loss` broken out by cache capacity.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for cap in sorted(sample_df["capacity"].unique()):
    subset = sample_df.loc[sample_df["capacity"] == cap, "candidate_lru_score"]
    axes[0].hist(subset, bins=20, alpha=0.55, label=f"capacity={cap}")
axes[0].set_xlabel("candidate_lru_score")
axes[0].set_ylabel("count (sample)")
axes[0].set_title("LRU-score distribution by capacity")
axes[0].legend(fontsize=8)

mean_loss_by_cap = sample_df.groupby("capacity")["y_loss"].mean()
axes[1].bar(mean_loss_by_cap.index.astype(str), mean_loss_by_cap.values)
axes[1].axhline(sample_df["horizon"].iloc[0], color="gray", linestyle="--", label="horizon")
axes[1].set_xlabel("Cache capacity")
axes[1].set_ylabel("mean y_loss (sample)")
axes[1].set_title("Mean y_loss by capacity (sample)")
axes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()


## What is one question that you have answered using these data?

**"Do the predictor-informed and LRU eviction policies actually disagree
often, and can engineered candidate features distinguish the cases where
they do?"** The dataset's `predictor_lru_disagree` indicator column and the
`*_gap_to_predictor_best` / `*_gap_to_lru_victim` feature family were built
specifically to support this question, and the released research (see
`Citation` below) uses exactly these columns to study predictor-vs-LRU
disagreement and its relationship to eviction outcomes. This is a fully
reproducible example: load the full `cross_family_evict_value_v1`
configuration, group by `predictor_lru_disagree`, and compare the `y_loss`
distribution in each group.


## What is one unanswered question, or a challenge for the community?

**This one came directly out of preparing this notebook, not out of prior
analysis** — which is exactly the kind of question this section is meant
to surface: In every stratified sample pulled while preparing this
notebook (spanning three widely separated row groups and all three cache
capacities), `y_loss` was **exactly 4.0 for every single row** — i.e., it
was saturated at the finite horizon value in 100% of the ~2,400 rows
sampled. Is `y_loss` saturation this common across the *entire* 11.2M-row
configuration, or is this an artifact of how the file's row groups happen
to be laid out? If saturation really is this pervasive, what does that
imply about `horizon = 4` as a modeling choice, and would a
longer-horizon label (from `objective_ablation_scalar`'s censored fields,
or a hypothetical future release) carry meaningfully more signal for
policy learning? We would welcome a community analysis of the full-file
`y_loss` distribution, stratified by capacity and by `decision_chunk_id`,
as a first step toward answering this.


## Reproducibility notes

- The release was produced by a deterministic pipeline (derived access
  sequence constructed from Wikimedia pageview popularity data -> cache
  simulation -> per-candidate feature/label computation ->
  deterministic pseudonymization) applied to the full available set of
  `wiki2018` source shards, without down-sampling.
- Generator source code (cache simulation, feature/label computation):
  https://github.com/SoroushVahidi/Augmented-caching, branch `main`. This
  notebook's own repository packages an already-generated candidate-row
  tree; it does not regenerate rows from raw traces.
- A SHA-256 checksum manifest and a schema-validation report ship inside
  the release's `metadata/` directory alongside the Parquet files.
- This notebook's own sampling is **not** a uniform random sample of the
  full 11.2M rows -- it reads three specific row groups (see code above) and
  is meant for orientation, not for drawing statistical conclusions about
  the full dataset. Use the full configuration (or a proper random sample
  across all row groups) for any real analysis.

## Limitations

- No held-out split -- every row is labeled `split = "train"`.
- Unbalanced capacity distribution (~14.3% / 28.6% / 57.1% for 32/64/128).
- `y_loss` is a finite-horizon counterfactual under LRU continuation, not a
  globally optimal control target -- see the open question above regarding
  how often it saturates at the horizon.
- Single source family (`wiki2018` only) in this release -- no
  cross-workload generalization claim should be drawn from this release alone.

## Citation and attribution

```bibtex
@dataset{vahidi_lafc_evict_v0_3,
  author    = {Vahidi, Soroush},
  title     = {LAFC-Evict: Learning-Augmented Cache Eviction Dataset (v0.3)},
  year      = {2026},
  publisher = {Hugging Face},
  url       = {https://huggingface.co/datasets/SoroushVahidi/lafc-evict}
}
```

```bibtex
@article{vahidi_decision_aligned_eviction,
  author = {Vahidi, Soroush},
  title  = {Decision-aligned eviction-value prediction for robust learning-augmented caching},
  year   = {2026},
  note   = {Public SSRN preprint, abstract ID 6636732; no peer-reviewed venue claimed},
  url    = {https://ssrn.com/abstract=6636732}
}
```

Upstream Wikimedia pageview data (CC0-1.0, `dumps.wikimedia.org/other/pageviews/`)
should be cited separately by anyone using it directly; it is not
redistributed by this dataset.

**Contact:** sv96@njit.edu


---
**DATA PROVIDER: PLEASE REMEMBER TO CLEAR ALL OUTPUTS FROM THIS NOTEBOOK
BEFORE COMMITTING/PUBLISHING**, per the AWS "Get To Know A Dataset"
template's own reviewer note. This copy lives in a private AWS Open Data
review repository; it has not yet been committed to a public GitHub
repository, which AWS's onboarding process requires before Registry
launch. AWS Open Data Step 5 (CloudFormation resource creation and public
S3 upload) is complete — this notebook now reads directly from the real,
public `lafc-evict-open-data` S3 bucket rather than a placeholder. The
only remaining placeholder is `<PENDING_REGISTRY_LANDING_PAGE_URL>` in the
title cell above, which depends on the Registry entry going live. See
`../validation/TUTORIAL_VALIDATION.md` for exactly which cells were
executed during preparation.
